# 🔴 Redis for AI — Cal Hacks Workshop

> **Time:** ~40 minutes &nbsp;|&nbsp; **Level:** beginner-friendly

In this workshop you'll build a knowledge-grounded AI chatbot backed entirely by Redis. Along the way you'll see three Redis superpowers that show up in almost every production AI app:

| Part | Feature | What it does |
|------|---------|-------------|
| 1 | **Vector Search** | Store and search knowledge by *meaning*, not keywords |
| 2 | **Semantic Cache** | Skip repeated LLM calls — instant answers for similar questions |
| 3 | **Session Memory** | Give your chatbot a memory so it remembers the conversation |

By the end you'll have a working **Hack Buddy** — an AI assistant that knows about Redis, AI patterns, and hackathon projects — ready to remix for your own hack.

## 🛠️ Setup

### 1. Install packages

In [1]:
%pip install -q "openai>=1.68" "redis>=5.2" "redisvl[langcache]>=0.16" "python-dotenv>=1.0" "agent-memory-client>=0.14" numpy


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 2. Configure credentials

Create a `.env` file in the same folder as this notebook:

```
# OpenAI
OPENAI_API_KEY=sk-...

# Redis Cloud
REDIS_URL=redis://default:<password>@<host>:<port>

# Redis Agent Memory Server
MEMORY_BASE_URL=https://<your-memory-server>
```

> **Redis Agent Memory** is an open-source server by Redis that adds two-tier memory (session + long-term) to any AI app. For the workshop, the server URL will be on the whiteboard. To run your own: `docker run -p 8000:8000 redislabs/agent-memory-server`

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

REDIS_URL      = os.environ['REDIS_URL']
OPENAI_KEY     = os.environ['OPENAI_API_KEY']
MEMORY_URL     = os.environ.get('MEMORY_BASE_URL', 'http://localhost:8000')

EMBED_MODEL    = 'text-embedding-3-small'
CHAT_MODEL     = 'gpt-4o-mini'
USER_ID        = 'workshop-user-001'

print('Credentials loaded')

KeyError: 'REDIS_URL'

In [ ]:
import json, time, uuid
import numpy as np
import redis
from openai import OpenAI
from redisvl.index import SearchIndex
from redisvl.query import VectorQuery
from redisvl.extensions.cache.llm import LangCacheSemanticCache
from agent_memory_client import MemoryAPIClient, MemoryClientConfig
from agent_memory_client.models import (
    WorkingMemory, MemoryMessage, ClientMemoryRecord, MemoryTypeEnum
)

openai_client = OpenAI(api_key=OPENAI_KEY)
redis_client = redis.from_url(REDIS_URL, decode_responses=True)

redis_client.ping()  # will raise an error if connection fails

def embed(text: str) -> list[float]:
    """Return an embedding vector for text."""
    return openai_client.embeddings.create(input=text, model=EMBED_MODEL).data[0].embedding

print('Connected to Redis and OpenAI')

---
## 🔍 Part 1 — Redis as a Vector Database

### How it works

1. Each document is turned into an **embedding** — a list of numbers that captures its *meaning*.
2. Embeddings are stored in Redis alongside the original text.
3. At query time, your question is embedded the same way and Redis finds the closest matches.

This is the foundation of **RAG (Retrieval-Augmented Generation)**: retrieve relevant facts from Redis, then hand them to the LLM as context.

```
User question ──► embed ──► Redis vector search ──► top-k docs ──► LLM ──► answer
```

In [ ]:
# Our knowledge base — short descriptions of Redis + AI concepts.
# In a real app these might come from your docs, a database, or scraped web pages.

KNOWLEDGE_BASE = [
    {"id": "kb:01", "title": "Vector Search",
     "text": "Redis Vector Search lets you store embedding vectors alongside your data "
             "and run nearest-neighbour queries in milliseconds. Use it to build semantic "
             "search, recommendation engines, and RAG pipelines."},
    {"id": "kb:02", "title": "Semantic Caching",
     "text": "LangCache is Redis's semantic cache for LLM responses. Instead of caching "
             "by exact string, it caches by meaning — so a paraphrase of a cached question "
             "gets an instant answer without hitting the LLM API."},
    {"id": "kb:03", "title": "Session Memory",
     "text": "Store chat history in Redis lists so your AI agent remembers the current "
             "conversation. Each message is pushed to a keyed list with a TTL so memory "
             "auto-expires when the session ends."},
    {"id": "kb:04", "title": "Long-Term Memory",
     "text": "Persist user preferences and past interactions as embedding vectors. When a "
             "user returns, semantic search surfaces the most relevant memories to personalise "
             "the response."},
    {"id": "kb:05", "title": "RAG (Retrieval-Augmented Generation)",
     "text": "RAG grounds LLM responses in real data. Embed your documents, store them in "
             "Redis, and retrieve the top-k matches at query time. Pass the retrieved text to "
             "the LLM as context so it answers from your data, not just its training."},
    {"id": "kb:06", "title": "Rate Limiting LLM APIs",
     "text": "Use Redis INCR + EXPIRE to enforce per-user token or request limits against "
             "your LLM API. A sliding-window counter in Redis prevents runaway costs and "
             "ensures fair usage across users."},
    {"id": "kb:07", "title": "Redis Streams for AI Pipelines",
     "text": "Redis Streams act as a durable message queue between AI pipeline stages — "
             "ingestion, embedding, storage, retrieval. Each stage reads from a stream, "
             "processes, and writes to the next, giving you backpressure and replay for free."},
    {"id": "kb:08", "title": "Pub/Sub for Multi-Agent Systems",
     "text": "Redis Pub/Sub lets AI agents broadcast events to each other in real time. "
             "One agent publishes a result; others subscribe and react — great for "
             "orchestrating parallel tool-calling agents."},
    {"id": "kb:09", "title": "Leaderboards and Rankings",
     "text": "Redis Sorted Sets are perfect for AI app leaderboards: store user scores or "
             "model accuracy metrics with ZADD and retrieve top-N with ZRANGE in O(log N). "
             "Ideal for hackathon judging dashboards."},
    {"id": "kb:10", "title": "JSON Storage for AI Outputs",
     "text": "Redis JSON lets you store, query, and update structured AI outputs — tool call "
             "results, agent state, model metadata — without serialising to a string. "
             "JSONPath queries let you filter nested fields directly."},
]

print(f'Knowledge base: {len(KNOWLEDGE_BASE)} documents')

In [ ]:
# Define the Redis index schema — one vector field + two text fields.
SCHEMA = {
    "index": {"name": "hack-buddy-kb", "prefix": "kb"},
    "fields": [
        {"name": "title",  "type": "text"},
        {"name": "text",   "type": "text"},
        {
            "name": "embedding",
            "type": "vector",
            "attrs": {
                "algorithm": "flat",
                "dims": 1536,
                "distance_metric": "cosine",
                "datatype": "float32",
            },
        },
    ],
}

index = SearchIndex.from_dict(SCHEMA)
index.set_client(redis_client)
index.create(overwrite=True)

# Embed each document and load into Redis
records = []
for doc in KNOWLEDGE_BASE:
    vec = embed(doc['text'])
    records.append({
        **doc,
        'embedding': np.array(vec, dtype=np.float32).tobytes(),
    })

index.load(records, id_field='id')
print(f'✅ Indexed {len(records)} documents')

In [ ]:
def search_knowledgebase(question: str, top_k: int = 3) -> list[dict]:
    """Semantic search: return the top_k most relevant documents."""
    q = VectorQuery(
        vector=embed(question),
        vector_field_name='embedding',
        return_fields=['title', 'text'],
        num_results=top_k,
    )
    return index.query(q)

# Try it — note it finds relevant docs even with different wording
results = search_knowledgebase('How do I make my chatbot remember past messages?')
for search_result in results:
    print(f"  [{search_result['vector_distance']:.3f}] {search_result['title']}")
    print(f"          {search_result['text'][:80]}...\n")

In [ ]:
# Try a different question — notice the different docs surface
results2 = search_knowledgebase('speed up my app and reduce API costs')
for search_result in results2:
    print(f"  [{search_result['vector_distance']:.3f}] {search_result['title']}")
    print(f"          {search_result['text'][:80]}...\n")

---
## ⚡ Part 2 — Semantic Caching

LLM API calls are slow (~1–3 s) and expensive. **Semantic caching** solves this by caching responses based on *meaning* rather than the exact string.

If someone asks *"What is the capital of France?"* and the cache holds the answer to *"Tell me France's capital city"*, the second user gets an **instant response** — no API call.

```
Question ──► embed ──► Redis similarity check
                              │
                    ┌─────────┴──────────┐
                  HIT ✅               MISS ❌
                    │                    │
              return cached         call LLM
              response              store result
```

In [ ]:
cache = LangCacheSemanticCache(
    name='hack-buddy-cache',
    redis_url=REDIS_URL,
    distance_threshold=0.15,   # lower = stricter matching
)
cache.clear()  # start fresh for the demo
print('LangCache ready')

In [ ]:
def ask_with_cache(question: str) -> tuple[str, float, bool]:
    """Ask a question, using the cache when possible. Returns (answer, elapsed, cache_hit)."""
    start = time.time()

    # 1. Check cache first
    hits = cache.check(prompt=question, num_results=1)
    if hits:
        elapsed = time.time() - start
        return hits[0]['response'], elapsed, True

    # 2. Cache miss — call the LLM
    relevant_documents = search_knowledgebase(question, top_k=2)
    context = '\n'.join(f"- {document['title']}: {document['text']}" for document in relevant_documents)
    chat_response = openai_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": "Answer concisely using the context below.\n\n" + context},
            {"role": "user",   "content": question},
        ],
        temperature=0.2,
    )
    answer = chat_response.choices[0].message.content.strip()

    # 3. Store in cache for next time
    cache.store(prompt=question, response=answer)

    elapsed = time.time() - start
    return answer, elapsed, False

# First call — cache miss, hits the LLM
Q1 = 'How does Redis vector search work?'
answer, t, hit = ask_with_cache(Q1)
print(f'Cache HIT: {hit}  |  Time: {t:.2f}s')
print(f'Answer: {answer[:200]}')

In [ ]:
# Paraphrase of the same question — should be a cache HIT
Q2 = 'Can you explain how vector similarity search works in Redis?'
answer2, t2, hit2 = ask_with_cache(Q2)
print(f'Cache HIT: {hit2}  |  Time: {t2:.4f}s  ← much faster!')
print(f'Answer: {answer2[:200]}')

---
## 🧠 Part 3 — Agent Memory

Without memory, every message is the first. **Redis Agent Memory** gives your chatbot two layers:

| Layer | What it stores | Scope |
|-------|---------------|-------|
| **Working memory** | Current conversation messages | Per session (expires) |
| **Long-term memory** | Facts, preferences, important context | Persistent across sessions |

Both are powered by Redis under the hood. We access them through the `agent-memory-client` Python package, which talks to the [Redis Agent Memory Server](https://github.com/redis/agent-memory-server).

```python
from agent_memory_client import MemoryAPIClient, MemoryClientConfig
```

In [ ]:
memory_client = MemoryAPIClient(
    MemoryClientConfig(
        base_url=MEMORY_URL,
        default_namespace='hack-buddy',
    )
)

# Seed a long-term memory — facts the bot should always know about this user
await memory_client.create_long_term_memory([
    ClientMemoryRecord(
        text='User is attending Cal Hacks and building an AI project with Redis.',
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=['hackathon', 'redis', 'context'],
        user_id=USER_ID,
    ),
    ClientMemoryRecord(
        text='User prefers Python and is comfortable with async code.',
        memory_type=MemoryTypeEnum.SEMANTIC,
        topics=['programming', 'preferences'],
        user_id=USER_ID,
    ),
])
print('Long-term memories seeded')

In [ ]:
# Search long-term memory semantically
results = await memory_client.search_long_term_memory(
    text='What does the user know about programming?',
    limit=3,
)
print('Long-term memory search results:')
for memory in results.memories:
    score = 1 - memory.dist if memory.dist else 0
    print(f'  [{score:.2f}] {memory.text}')

# Working memory — session history
demo_session = f'demo-{uuid.uuid4().hex[:6]}'
working_memory = WorkingMemory(
    session_id=demo_session,
    messages=[
        MemoryMessage(role='user',      content='I am building a music recommender.'),
        MemoryMessage(role='assistant', content='Nice! Redis vector search is a great fit.'),
        MemoryMessage(role='user',      content='Should I use cosine or dot-product?'),
    ],
)
await memory_client.put_working_memory(demo_session, working_memory)

_, saved_working_memory = await memory_client.get_or_create_working_memory(session_id=demo_session)
print(f'\nWorking memory: {len(saved_working_memory.messages)} message(s) in session')
for message in saved_working_memory.messages:
    print(f'  {message.role.upper():12} {message.content}')

---
## 🤖 Part 4 — Hack Buddy: the Full Pipeline

Now we combine all three pieces:

```
 User message
      │
      ▼
 ① Semantic Cache ──HIT──► instant answer
      │ MISS
      ▼
 ② Vector Search  ──────► retrieve relevant docs
      │
      ▼
 ③ Session Memory ──────► prepend conversation history
      │
      ▼
   LLM call  ──────────► answer
      │
      ▼
 store in memory + cache
```

In [ ]:
async def chat(message: str, session_id: str, use_cache: bool = True) -> str:
    """One turn of Hack Buddy — vector search + agent memory + semantic cache."""
    message = message.strip()

    # ① Semantic cache check
    if use_cache:
        hits = cache.check(prompt=message, num_results=1)
        if hits:
            print('  [cache HIT]')
            return hits[0]['response']
    print('  [cache miss — calling LLM]')

    # ② Load session history from working memory
    _, working_memory = await memory_client.get_or_create_working_memory(
        session_id=session_id, user_id=USER_ID)
    history = [{'role': message.role, 'content': message.content} for message in working_memory.messages[-8:]]

    # ③ Retrieve relevant knowledge via vector search
    relevant_documents = search_knowledgebase(message, top_k=3)
    context = '\n'.join(f"- {document['title']}: {document['text']}" for document in relevant_documents)

    # ④ Build messages and call LLM
    system = (
        'You are Hack Buddy, a helpful assistant for hackathon developers. '
        'Answer concisely using the knowledge below. If unsure, say so.\n\n'
        f'Knowledge:\n{context}'
    )
    messages = [{'role': 'system', 'content': system}]
    messages.extend(history)
    messages.append({'role': 'user', 'content': message})

    chat_response = openai_client.chat.completions.create(model=CHAT_MODEL, messages=messages, temperature=0.3)
    answer = chat_response.choices[0].message.content.strip()

    # ⑤ Persist to working memory and cache
    updated_working_memory = WorkingMemory(
        session_id=session_id,
        messages=list(working_memory.messages) + [
            MemoryMessage(role='user',      content=message),
            MemoryMessage(role='assistant', content=answer),
        ],
    )
    await memory_client.put_working_memory(session_id, updated_working_memory)
    cache.store(prompt=message, response=answer)

    return answer


SESSION = f'hack-buddy-{uuid.uuid4().hex[:6]}'
print('Hack Buddy ready — session:', SESSION)

In [ ]:
# Turn 1 — general question
q1 = 'What Redis feature should I use for a recommendation system?'
print(f'You: {q1}')
a1 = chat(q1, SESSION)
print(f'Hack Buddy: {a1}\n')

In [ ]:
# Turn 2 — follow-up; the model now has the previous turn in context
q2 = 'How many dimensions should my embeddings be?'
print(f'You: {q2}')
a2 = chat(q2, SESSION)
print(f'Hack Buddy: {a2}\n')

In [ ]:
# Turn 3 — repeat a similar question to trigger the cache
q3 = 'What is semantic similarity search in Redis?'
print(f'You: {q3}')
a3 = chat(q3, SESSION)
print(f'Hack Buddy: {a3}\n')

---
## 🚀 What to Build Next

You now have all three primitives. Here are some hackathon ideas that combine them:

- **Code Review Bot** — embed your codebase, answer questions about it, remember context
- **Event Recommender** — embed event descriptions, find what matches a user's vibe
- **Study Buddy** — RAG over lecture notes + long-term memory per student
- **Multi-Agent Orchestrator** — agents communicate via Redis Pub/Sub and store state in JSON

### Useful links
- [Redis AI docs](https://redis.io/docs/latest/develop/ai/)
- [RedisVL GitHub](https://github.com/redis/redis-vl-python)
- [LangCache docs](https://redis.io/docs/latest/develop/ai/langcache/)
- [Redis Cloud free tier](https://redis.io/try-free)

> **Need help?** Find us at the Redis booth or ping `@redis` in the Discord.